In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Memantau atau membatalkan job

Lihat daftar beban kerja kamu di [halaman Workloads](https://quantum.cloud.ibm.com/workloads).

## Lihat status job
Buka [tabel Workloads](https://quantum.cloud.ibm.com/workloads) dan periksa kolom Status untuk melihat apakah job sudah selesai atau gagal.

## Lihat sisa penggunaan
Buka [tabel Instances](https://quantum.cloud.ibm.com/instances) dan pilih tab yang sesuai dengan paket yang ingin kamu lihat sisa penggunaannya. Total waktu yang sudah digunakan dan total waktu yang tersisa pada paket kamu akan ditampilkan.

## Lihat metrik jumlah job dan workload yang dikirimkan
Buka [halaman Analytics](https://quantum.cloud.ibm.com/analytics) untuk melihat total jumlah job yang dikirimkan, serta jumlah batch workload dan session workload. Perlu diingat bahwa kamu hanya bisa melihat halaman Analytics untuk akun yang kamu miliki atau kelola.

## Pantau job
Gunakan instance job untuk memeriksa status job atau mengambil hasilnya dengan memanggil perintah yang sesuai:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Tinjau hasil job segera setelah job selesai. Hasil job tersedia setelah job selesai. Oleh karena itu, job.result() adalah panggilan pemblokiran sampai job selesai.                               |
| job.job\_id()                 | Kembalikan ID yang secara unik mengidentifikasi job tersebut. Mengambil hasil job di lain waktu membutuhkan ID job. Oleh karena itu, disarankan untuk menyimpan ID job yang mungkin ingin kamu ambil nanti. |
| job.status()                  | Periksa status job.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Ambil job yang sebelumnya kamu kirimkan. Panggilan ini membutuhkan ID job.                                                                                                                                      |

<span id="retrieve-later"></span>
## Ambil hasil job di lain waktu
Panggil `service.job(\<job\_id>)` untuk mengambil job yang sebelumnya kamu kirimkan. Jika kamu tidak punya ID job, atau jika ingin mengambil beberapa job sekaligus — termasuk job dari QPU (unit pemrosesan kuantum) yang sudah pensiun — panggil `service.jobs()` dengan filter opsional. Lihat [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` juga mengembalikan job yang dijalankan dari paket `qiskit-ibm-provider` yang sudah tidak didukung. Job yang dikirimkan oleh paket yang lebih lama (juga sudah tidak didukung) `qiskit-ibmq-provider` tidak lagi tersedia.

### Contoh
Contoh ini mengembalikan 10 runtime job terbaru yang dijalankan di `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>